In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
# Some lists!

todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [10]:
get_todo_report()

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [12]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [13]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [14]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [15]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [16]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [17]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [18]:
todos, completed = [], []
loop(messages)

Todo #1: Convert departure times to a common reference (e.g., hours after a certain time).
Todo #2: Calculate the head start distance covered by the first train before the second train departs.
Todo #3: Determine the relative speed of the two trains when they are moving towards each other.
Todo #4: Calculate the time until the two trains meet after the second train departs.
Todo #5: Find the actual meeting time based on the departure times.

Todo #1: Convert departure times to a common reference (e.g., hours after a certain time).
Todo #2: Calculate the head start distance covered by the first train before the second train departs.
Todo #3: Determine the relative speed of the two trains when they are moving towards each other.
Todo #4: Calculate the time until the two trains meet after the second train departs.
Todo #5: Find the actual meeting time based on the departure times.
Todo #6: Estimate the distance between Boston and New York (roughly 215 miles).

Converted 2:00 pm to hours after 2:00 pm as 0 hours, and 3:00 pm as 1 hour after 2:00 pm.

Todo #1: Convert departure times to a common reference (e.g., hours after a certain time).
Todo #2: Calculate the head start distance covered by the first train before the second train departs.
Todo #3: Determine the relative speed of the two trains when they are moving towards each other.
Todo #4: Calculate the time until the two trains meet after the second train departs.
Todo #5: Find the actual meeting time based on the departure times.
Todo #6: Estimate the distance between Boston and New York (roughly 215 miles).

In the first hour, the first train travels 60 miles (since 60 mph for 1 hour).

...

Time until they meet after the second train departs is distance / relative speed = 215 miles / 140 mph = 1.5357 
hours.

Todo #1: Convert departure times to a common reference (e.g., hours after a certain time).
Todo #2: Calculate the head start distance covered by the first train before the second train departs.
Todo #3: Determine the relative speed of the two trains when they are moving towards each other.
Todo #4: Calculate the time until the two trains meet after the second train departs.
Todo #5: Find the actual meeting time based on the departure times.
Todo #6: Estimate the distance between Boston and New York (roughly 215 miles).
Todo #7: Calculate the relative speed of the two trains when they are moving towards each other.
Todo #8: Calculate the distance between Boston and New York to get a more precise estimate, if necessary.
Todo #9: Determine how long it takes for the two trains to meet after the second train starts.
Todo #10: Calculate the exact meeting time in hours after the initial reference point, then convert to actual time.

Calculate the actual meeting time in hours after the initial reference point is 2.0 hours (departure of the first 
train at 2:00 pm) plus 1.5357 hours (from the second train start), totaling approximately 3.5357 hours after 2:00 
pm. This corresponds to about 5:32 pm.

Todo #1: Convert departure times to a common reference (e.g., hours after a certain time).
Todo #2: Calculate the head start distance covered by the first train before the second train departs.
Todo #3: Determine the relative speed of the two trains when they are moving towards each other.
Todo #4: Calculate the time until the two trains meet after the second train departs.
Todo #5: Find the actual meeting time based on the departure times.
Todo #6: Estimate the distance between Boston and New York (roughly 215 miles).
Todo #7: Calculate the relative speed of the two trains when they are moving towards each other.
Todo #8: Calculate the distance between Boston and New York to get a more precise estimate, if necessary.
Todo #9: Determine how long it takes for the two trains to meet after the second train starts.
Todo #10: Calculate the exact meeting time in hours after the initial reference point, then convert to actual time.

The two trains will meet approximately **3.54 hours after 2:00 pm**, which is around **5:32 pm**.